## Feautres - Akshara Selection, VQA & Binarization + Qwen VQA Integration

This version includes:
- **Binarized image display** (Otsu thresholding) - NEW!
- Automatic OCR with Tesseract
- Speech recognition for Hindi/English
- **Akshara Selection & Comparison**
- **VQA (Visual Question Answering)**
- Image similarity detection using MSE
- Image database for character instances
- Visual comparison dialog for similar aksharas


In [ ]:
import os
import io
import json
from PyQt5.QtWidgets import (QApplication, QMainWindow, QWidget, QVBoxLayout, 
                            QHBoxLayout, QLabel, QPushButton, QFileDialog, 
                            QScrollArea, QTextEdit, QSplitter, QSlider, 
                            QCheckBox, QMessageBox, QComboBox, QGroupBox,
                            QProgressBar, QShortcut, QLineEdit, QDialog, QGridLayout,
                            QRadioButton, QButtonGroup) 
from PyQt5.QtGui import QPixmap, QImage, QPainter, QPen, QColor, QFont, QIcon, QKeySequence
from PyQt5.QtCore import Qt, QThread, pyqtSignal, QRect, QPoint, QTimer, QBuffer, QIODevice 
import fitz  # PyMuPDF
from PIL import Image
import numpy as np
import cv2
import sys
import speech_recognition as sr
import requests
import base64

# OCR imports - ADDED FOR OCR FUNCTIONALITY
try:
    import pytesseract
    TESSERACT_AVAILABLE = True
except ImportError:
    TESSERACT_AVAILABLE = False

# ==================== QWEN VL CLIENT ====================
class QwenVLClient:
    """Client for Qwen VL inference server"""
    def __init__(self, server_url="http://172.18.42.4:8080"):
        self.server_url = server_url.rstrip('/')
        self.infer_endpoint = f"{self.server_url}/infer"
        self.health_endpoint = f"{self.server_url}/health"
    
    def check_health(self):
        """Check if the server is healthy"""
        try:
            response = requests.get(self.health_endpoint, timeout=5)
            return response.json()
        except Exception as e:
            return {"status": "error", "message": str(e)}
    
    def image_to_base64(self, image_path):
        """Convert image file to base64 string"""
        with open(image_path, "rb") as image_file:
            encoded_string = base64.b64encode(image_file.read()).decode('utf-8')
        
        # Detect image format
        img = Image.open(image_path)
        img_format = img.format.lower() if img.format else 'jpeg'
        
        return f"data:image/{img_format};base64,{encoded_string}"
    
    def pil_to_base64(self, pil_image):
        """Convert PIL Image to base64 string"""
        buffered = io.BytesIO()
        pil_image.save(buffered, format="JPEG")
        encoded_string = base64.b64encode(buffered.getvalue()).decode('utf-8')
        return f"data:image/jpeg;base64,{encoded_string}"
    
    def qpixmap_to_base64(self, pixmap):
        """Convert QPixmap to base64 string"""
        # Convert QPixmap to QImage
        qimage = pixmap.toImage()
        
        # Save QImage to byte array using QBuffer
        buffer = QBuffer()
        buffer.open(QIODevice.WriteOnly)
        qimage.save(buffer, "PNG")
        
        # Get the data and encode to base64
        image_data = buffer.data()
        encoded_string = base64.b64encode(image_data).decode('utf-8')
        
        return f"data:image/png;base64,{encoded_string}"
    
    def infer(self, image_input, text_prompt, max_new_tokens=128):
        """
        Perform inference
        
        Args:
            image_input: Can be:
                - Path to image file (str)
                - PIL Image object
                - QPixmap object
                - Base64 encoded string
                - URL (str starting with http)
            text_prompt: Text prompt/question
            max_new_tokens: Maximum number of tokens to generate
        
        Returns:
            dict: Response containing output text or error
        """
        # Prepare image input
        if isinstance(image_input, str):
            if image_input.startswith("http"):
                # URL
                image_data = image_input
            elif image_input.startswith("data:image"):
                # Already base64
                image_data = image_input
            else:
                # File path
                image_data = self.image_to_base64(image_input)
        elif isinstance(image_input, Image.Image):
            # PIL Image
            image_data = self.pil_to_base64(image_input)
        elif isinstance(image_input, QPixmap):
            # QPixmap
            image_data = self.qpixmap_to_base64(image_input)
        else:
            raise ValueError("Unsupported image input type")
        
        # Prepare messages
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image_data},
                    {"type": "text", "text": text_prompt}
                ]
            }
        ]
        
        # Make request
        payload = {
            "messages": messages,
            "max_new_tokens": max_new_tokens
        }
        
        try:
            response = requests.post(
                self.infer_endpoint,
                json=payload,
                timeout=120  # 120 second timeout for inference
            )
            response.raise_for_status()
            return response.json()
        except requests.exceptions.Timeout:
            return {"success": False, "error": "Request timeout"}
        except requests.exceptions.RequestException as e:
            return {"success": False, "error": str(e)}
        except Exception as e:
            return {"success": False, "error": str(e)}

# ==================== QWEN VL WORKER ====================
class QwenVLWorker(QThread):
    """Worker thread to perform Qwen VL inference without freezing the GUI"""
    result_ready = pyqtSignal(str, str)  # answer, prompt
    error_occurred = pyqtSignal(str)
    progress_update = pyqtSignal(int, str)  # progress, status_message
    
    def __init__(self, pixmap, prompt, server_url="http://172.18.42.4:8080"):
        super().__init__()
        self.pixmap = pixmap
        self.prompt = prompt
        self.server_url = server_url
        self._is_running = True
        
    def run(self):
        """Perform Qwen VL inference"""
        try:
            self.progress_update.emit(10, "Connecting to Qwen VL server...")
            
            # Create client
            client = QwenVLClient(server_url=self.server_url)
            
            self.progress_update.emit(30, "Encoding image...")
            
            # Perform inference
            self.progress_update.emit(50, "Sending request to Qwen VL model...")
            
            result = client.infer(
                image_input=self.pixmap,
                text_prompt=self.prompt,
                max_new_tokens=256
            )
            
            self.progress_update.emit(90, "Processing response...")
            
            if result.get("success"):
                answer = result.get("output", "No response received")
                self.progress_update.emit(100, "Complete!")
                self.result_ready.emit(answer, self.prompt)
            else:
                error_msg = result.get("error", "Unknown error")
                self.error_occurred.emit(f"Inference error: {error_msg}")
                
        except Exception as e:
            self.error_occurred.emit(f"Qwen VL Error: {str(e)}")
    
    def stop(self):
        self._is_running = False

# ==================== IMAGE COMPARISON DIALOG ====================
class ImageComparisonDialog(QDialog):
    """Dialog to display similar images and choose insertion options"""
    
    def __init__(self, parent=None, selected_images=None, similar_images=None):
        super().__init__(parent)
        self.selected_images = selected_images or []
        self.similar_images = similar_images or []
        self.selected_similar_images = []
        self.is_insert_underscore = True  # Default to underscore
        self.init_ui()
        
    def init_ui(self):
        """Initialize the dialog UI"""
        self.setWindowTitle("Similar Aksharas")
        self.setMinimumWidth(700)
        self.setMinimumHeight(500)
        
        main_layout = QVBoxLayout()
        self.setLayout(main_layout)
        
        # Instructions
        instructions = QLabel("Your selections are highlighted with green borders. Select similar images to insert.")
        main_layout.addWidget(instructions)
        
        # Create a scroll area for the images grid
        scroll_area = QScrollArea()
        scroll_area.setWidgetResizable(True)
        scroll_content = QWidget()
        scroll_layout = QVBoxLayout(scroll_content)
        
        # Selected images section
        selected_group = QGroupBox("Your Selected Aksharas")
        selected_layout = QHBoxLayout()
        selected_group.setLayout(selected_layout)
        
        # Add all selected images with green border
        if self.selected_images:
            for selected_pixmap in self.selected_images:
                selected_container = QWidget()
                selected_container_layout = QVBoxLayout(selected_container)
                
                selected_label = QLabel()
                selected_label.setPixmap(selected_pixmap)
                selected_label.setStyleSheet("border: 3px solid green;")
                selected_label.setFixedSize(selected_pixmap.width() + 6, selected_pixmap.height() + 6)
                selected_container_layout.addWidget(selected_label)
                
                selected_layout.addWidget(selected_container)
        
        scroll_layout.addWidget(selected_group)
        
        # Similar images section
        similar_group = QGroupBox("Similar Aksharas Found")
        similar_layout = QGridLayout()
        similar_group.setLayout(similar_layout)
        
        # Add similar images with checkable frames
        self.image_buttons = []
        row, col = 0, 0
        max_cols = 5  # Maximum columns in the grid
        
        for i, (similarity, image, image_id) in enumerate(self.similar_images):
            # Create container for image and label
            image_container = QWidget()
            container_layout = QVBoxLayout(image_container)
            
            # Create checkable button for the image
            image_button = QPushButton()
            image_button.setCheckable(True)
            image_button.setFixedSize(image.width() + 10, image.height() + 10)
            image_button.setIcon(QIcon(image))
            image_button.setIconSize(image.size())
            image_button.clicked.connect(lambda checked, idx=i: self.toggle_image_selection(idx))
            container_layout.addWidget(image_button)
            
            # Add to grid layout
            similar_layout.addWidget(image_container, row, col)
            
            # Store button and image_id
            self.image_buttons.append((image_button, image_id))
            
            # Update grid position
            col += 1
            if col >= max_cols:
                col = 0
                row += 1
        
        scroll_layout.addWidget(similar_group)
        
        # Set scroll content and add to main layout
        scroll_area.setWidget(scroll_content)
        main_layout.addWidget(scroll_area)
        
        # Insertion options
        options_layout = QHBoxLayout()
        options_group = QButtonGroup(self)
        
        self.underscore_radio = QRadioButton("Insert underscore (_)")
        self.underscore_radio.setChecked(True)
        self.underscore_radio.toggled.connect(self.update_char_input_state)
        options_group.addButton(self.underscore_radio)
        options_layout.addWidget(self.underscore_radio)
        
        self.character_radio = QRadioButton("Insert akshara")
        self.character_radio.toggled.connect(self.update_char_input_state)
        options_group.addButton(self.character_radio)
        options_layout.addWidget(self.character_radio)
        
        # Add character input field
        options_layout.addWidget(QLabel("Akshara:"))
        self.char_input = QLineEdit()
        self.char_input.setMaxLength(5)
        self.char_input.setFixedWidth(60)
        self.char_input.setEnabled(False)
        options_layout.addWidget(self.char_input)
        
        main_layout.addLayout(options_layout)
        
        # Buttons
        buttons_layout = QHBoxLayout()
        
        self.cancel_button = QPushButton("Cancel")
        self.cancel_button.clicked.connect(self.reject)
        buttons_layout.addWidget(self.cancel_button)
        
        self.insert_button = QPushButton("Insert Selected")
        self.insert_button.clicked.connect(self.accept)
        buttons_layout.addWidget(self.insert_button)
        
        main_layout.addLayout(buttons_layout)

    def update_char_input_state(self):
        """Enable or disable the character input field based on radio button selection"""
        self.char_input.setEnabled(self.character_radio.isChecked())
        if self.character_radio.isChecked():
            self.char_input.setFocus()
    
    def toggle_image_selection(self, index):
        """Toggle image selection and update button styles"""
        button, image_id = self.image_buttons[index]
        if button.isChecked():
            button.setStyleSheet("background-color: lightblue;")
            if image_id not in self.selected_similar_images:
                self.selected_similar_images.append(image_id)
        else:
            button.setStyleSheet("")
            if image_id in self.selected_similar_images:
                self.selected_similar_images.remove(image_id)
    
    def get_results(self):
        """Get the dialog results"""
        is_insert_underscore = self.underscore_radio.isChecked()
        char_to_insert = "_"  # Default
        
        if not is_insert_underscore:
            # Get character from input field
            input_char = self.char_input.text()
            if input_char:
                char_to_insert = input_char
        
        return {
            'selected_images': self.selected_similar_images,
            'insert_underscore': is_insert_underscore,
            'character': char_to_insert
        }

# ==================== OCR WORKER ====================
class OCRWorker(QThread):
    """Worker thread to perform OCR without freezing the GUI"""
    result_ready = pyqtSignal(str, int, object)  # text, page_num, bbox
    error_occurred = pyqtSignal(str)
    progress_update = pyqtSignal(int)
    
    def __init__(self, pixmap, page_num, bbox=None, language='san'):
        super().__init__()
        self.pixmap = pixmap
        self.page_num = page_num
        self.bbox = bbox
        self.language = language
        self._is_running = True
        
    def run(self):
        """Perform OCR on the image"""
        try:
            if not TESSERACT_AVAILABLE:
                self.error_occurred.emit("Tesseract OCR is not installed")
                return
            
            self.progress_update.emit(10)
            
            # Convert QPixmap to PIL Image
            qimage = self.pixmap.toImage()
            width = qimage.width()
            height = qimage.height()
            ptr = qimage.bits()
            ptr.setsize(height * width * 4)
            arr = np.frombuffer(ptr, np.uint8).reshape((height, width, 4))
            pil_image = Image.fromarray(arr[:, :, :3])
            
            self.progress_update.emit(30)
            
            # If bbox is specified, crop the image
            if self.bbox and not self.bbox.isEmpty():
                pil_image = pil_image.crop((
                    self.bbox.x(),
                    self.bbox.y(),
                    self.bbox.x() + self.bbox.width(),
                    self.bbox.y() + self.bbox.height()
                ))
            
            self.progress_update.emit(50)
            
            # Preprocess image for better OCR
            img_array = np.array(pil_image)
            gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
            _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            denoised = cv2.fastNlMeansDenoising(binary)
            
            self.progress_update.emit(70)
            
            processed_image = Image.fromarray(denoised)
            
            # Perform OCR
            custom_config = r'--oem 3 --psm 6'
            text = pytesseract.image_to_string(
                processed_image,
                lang=self.language,
                config=custom_config
            )
            
            self.progress_update.emit(90)
            text = text.strip()
            self.progress_update.emit(100)
            
            self.result_ready.emit(text, self.page_num, self.bbox)
            
        except Exception as e:
            self.error_occurred.emit(f"OCR Error: {str(e)}")
    
    def stop(self):
        self._is_running = False

# ==================== SPEECH RECOGNITION WORKER ====================
class SpeechRecognitionWorker(QThread):
    """Worker thread to perform speech recognition without freezing the GUI"""
    result_ready = pyqtSignal(str)
    error_occurred = pyqtSignal(str)
    status_update = pyqtSignal(str)
    
    def __init__(self, language='hi-IN', timeout=5, phrase_time_limit=None):
        super().__init__()
        self.language = language
        self.timeout = timeout
        self.phrase_time_limit = phrase_time_limit
        self.listening = False
        self.recognizer = sr.Recognizer()
        
    def run(self):
        """Start listening and recognition in a separate thread"""
        try:
            self.listening = True
            self.status_update.emit("Listening...")
            
            with sr.Microphone() as source:
                self.recognizer.adjust_for_ambient_noise(source, duration=1)
                self.status_update.emit("Speak now...")
                
                try:
                    audio = self.recognizer.listen(
                        source, 
                        timeout=self.timeout,
                        phrase_time_limit=self.phrase_time_limit
                    )
                    
                    self.status_update.emit("Processing speech...")
                    
                    text = self.recognizer.recognize_google(audio, language=self.language)
                    self.result_ready.emit(text)
                except sr.WaitTimeoutError:
                    self.error_occurred.emit("No speech detected within timeout period")
                except sr.UnknownValueError:
                    self.error_occurred.emit("Could not understand audio")
                except sr.RequestError as e:
                    self.error_occurred.emit(f"Could not request results; {str(e)}")
        except Exception as e:
            self.error_occurred.emit(f"Error in speech recognition: {str(e)}")
        finally:
            self.listening = False

# ==================== PDF VIEW LABEL ====================
class PDFViewLabel(QLabel):
    """Custom label for PDF viewing and selection"""
    selectionChanged = pyqtSignal(QRect)
    selectionFinished = pyqtSignal(QRect)
    
    def __init__(self, parent=None):
        super().__init__(parent)
        self.selection_mode = False
        self.drawing = False
        self.start_point = QPoint()
        self.current_rect = QRect()
        self.bounding_boxes = []
        self.user_selections = []
        self.setCursor(Qt.ArrowCursor)
    
    def enable_selection(self, enabled):
        self.selection_mode = enabled
        if enabled:
            self.setCursor(Qt.CrossCursor)
        else:
            self.setCursor(Qt.ArrowCursor)
            self.current_rect = QRect()
            self.update()
    
    def set_bounding_boxes(self, boxes):
        """Set watershed algorithm bounding boxes"""
        self.bounding_boxes = boxes
        self.update()
    
    def clear_bounding_boxes(self):
        """Clear watershed algorithm bounding boxes"""
        self.bounding_boxes = []
        self.update()

    def add_user_selection(self, rect):
        """Add a user selection to the list of selections"""
        if not rect.isEmpty():
            self.user_selections.append(rect)
            self.update()
            return True
        return False
    
    def clear_user_selections(self):
        """Clear all user selections"""
        self.user_selections = []
        self.update()
    
    def mousePressEvent(self, event):
        if self.selection_mode and event.button() == Qt.LeftButton:
            self.drawing = True
            self.start_point = event.pos()
            self.current_rect = QRect()
            self.update()
        elif not self.selection_mode and event.button() == Qt.LeftButton:
            for box in self.bounding_boxes:
                if box.contains(event.pos()):
                    self.current_rect = box
                    self.selectionFinished.emit(box)
                    self.update()
                    break
    
    def mouseMoveEvent(self, event):
        if self.selection_mode and self.drawing:
            self.current_rect = QRect(self.start_point, event.pos()).normalized()
            self.selectionChanged.emit(self.current_rect)
            self.update()
    
    def mouseReleaseEvent(self, event):
        if self.selection_mode and self.drawing and event.button() == Qt.LeftButton:
            self.drawing = False
            self.current_rect = QRect(self.start_point, event.pos()).normalized()
            self.selectionFinished.emit(self.current_rect)
            self.update()
    
    def paintEvent(self, event):
        super().paintEvent(event)
        
        painter = QPainter(self)
        
        # Draw watershed bounding boxes with blue color
        if self.bounding_boxes:
            pen = QPen(QColor(0, 0, 255, 120), 1)
            painter.setPen(pen)
            
            for box in self.bounding_boxes:
                if box == self.current_rect:
                    painter.setPen(QPen(QColor(0, 180, 0, 180), 2))
                    painter.drawRect(box)
                    painter.fillRect(box, QColor(0, 180, 0, 30))
                    painter.setPen(pen)
                else:
                    painter.drawRect(box)
        
        # Draw user selections with green color
        if self.user_selections:
            pen = QPen(QColor(0, 180, 0, 180), 2)
            painter.setPen(pen)
            
            for box in self.user_selections:
                painter.drawRect(box)
                painter.fillRect(box, QColor(0, 180, 0, 30))
        
        # Draw manual selection with red color
        if self.selection_mode and not self.current_rect.isEmpty():
            pen = QPen(QColor(255, 0, 0, 180), 2)
            painter.setPen(pen)
            painter.drawRect(self.current_rect)
            painter.fillRect(self.current_rect, QColor(255, 0, 0, 60))


# ==================== MAIN APPLICATION ====================
class PDFOCRViewer(QMainWindow):
    def __init__(self):
        super().__init__()
        self.init_ui()
        self.pdf_document = None
        self.current_page = 0
        self.total_pages = 0
        self.watershed_workers = []
        self.zoom_factor = 1.0
        self.curr_page_pixmap = None
        self.current_selection = None
        self.current_pdf_filename = ""
        
        # OCR variables
        self.ocr_workers = []
        self.ocr_results = {}
        self.ocr_language = 'san'

        # Qwen VL variables
        self.qwen_workers = []
        self.qwen_server_url = "http://172.18.42.4:8080"
        
        # AKSHARA SELECTION: Initialize image lookup data
        self.image_lookup_dir = "Image_Lookup"
        self.image_lookup_data = {}
        self.next_image_id = 0
        self.load_image_lookup_data()
        
        # Initialize text editor context menu
        self.init_text_editor_context_menu()
        
        # Initialize speech recognition
        self.init_speech_recognition()
        
    def init_ui(self):
        self.setWindowTitle("PANDULIPI")
        self.setGeometry(100, 100, 1200, 800)
        
        # Main widget and layout
        main_widget = QWidget()
        main_layout = QVBoxLayout()
        self.main_layout = main_layout
        main_widget.setLayout(main_layout)
        self.setCentralWidget(main_widget)
        
        # Create a horizontal layout for PDF and Zoom controls
        controls_horizontal = QHBoxLayout()
        
        # Load PDF Group Box
        pdf_group = QGroupBox("PDF Controls")
        pdf_layout = QHBoxLayout()
        pdf_layout.setContentsMargins(10, 5, 10, 5)
        pdf_group.setLayout(pdf_layout)
        
        self.load_button = QPushButton("Load Manuscript")
        self.load_button.clicked.connect(self.load_pdf)
        pdf_layout.addWidget(self.load_button)
        
        self.prev_button = QPushButton("Previous Page")
        self.prev_button.clicked.connect(self.prev_page)
        self.prev_button.setEnabled(False)
        pdf_layout.addWidget(self.prev_button)
        
        self.page_label = QLabel("Page 0 of 0")
        pdf_layout.addWidget(self.page_label)
        
        self.next_button = QPushButton("Next Page")
        self.next_button.clicked.connect(self.next_page)
        self.next_button.setEnabled(False)
        pdf_layout.addWidget(self.next_button)
        
        pdf_layout.addStretch()
        controls_horizontal.addWidget(pdf_group)
        
        # Zoom Group Box
        zoom_group = QGroupBox("Zoom Controls")
        zoom_layout = QHBoxLayout()
        zoom_layout.setContentsMargins(10, 5, 10, 5)
        zoom_group.setLayout(zoom_layout)
        
        self.zoom_out_button = QPushButton("Zoom Out")
        self.zoom_out_button.clicked.connect(self.zoom_out)
        self.zoom_out_button.setEnabled(False)
        zoom_layout.addWidget(self.zoom_out_button)
        
        self.zoom_slider = QSlider(Qt.Horizontal)
        self.zoom_slider.setMinimum(10)
        self.zoom_slider.setMaximum(200)
        self.zoom_slider.setValue(35)
        self.zoom_slider.setTickInterval(10)
        self.zoom_slider.setTickPosition(QSlider.TicksBelow)
        self.zoom_slider.valueChanged.connect(self.zoom_slider_changed)
        self.zoom_slider.setEnabled(False)
        zoom_layout.addWidget(self.zoom_slider, 1)
        
        self.zoom_in_button = QPushButton("Zoom In")
        self.zoom_in_button.clicked.connect(self.zoom_in)
        self.zoom_in_button.setEnabled(False)
        zoom_layout.addWidget(self.zoom_in_button)
        
        self.zoom_label = QLabel("35%")
        zoom_layout.addWidget(self.zoom_label)
        
        self.zoom_reset_button = QPushButton("Reset Zoom")
        self.zoom_reset_button.clicked.connect(self.zoom_reset)
        self.zoom_reset_button.setEnabled(False)
        zoom_layout.addWidget(self.zoom_reset_button)

        # Add keyboard shortcuts for zoom
        self.zoom_in_shortcut = QShortcut(QKeySequence("Ctrl++"), self)
        self.zoom_in_shortcut.activated.connect(self.zoom_in)
        self.zoom_out_shortcut = QShortcut(QKeySequence("Ctrl+-"), self)
        self.zoom_out_shortcut.activated.connect(self.zoom_out)
        self.zoom_reset_shortcut = QShortcut(QKeySequence("Ctrl+0"), self)
        self.zoom_reset_shortcut.activated.connect(self.zoom_reset)

        self.zoom_in_button.setToolTip("Zoom In (Ctrl++)")
        self.zoom_out_button.setToolTip("Zoom Out (Ctrl+-)")
        self.zoom_reset_button.setToolTip("Reset Zoom (Ctrl+0)")
        
        controls_horizontal.addWidget(zoom_group)
        main_layout.addLayout(controls_horizontal)
        
        # Create second horizontal box
        controls_horizontal_ii = QHBoxLayout()

        # AKSHARA SELECTION: Selection Controls Group Box
        selection_group = QGroupBox("Akshara Selection & Annotation")
        selection_layout = QHBoxLayout()
        selection_layout.setContentsMargins(10, 5, 10, 5)
        selection_group.setLayout(selection_layout)
        
        self.selection_checkbox = QCheckBox("Enable Selection Mode")
        self.selection_checkbox.stateChanged.connect(self.toggle_selection_mode)
        self.selection_checkbox.setEnabled(False)
        selection_layout.addWidget(self.selection_checkbox)
        
        self.save_selections_button = QPushButton("Save & Compare Aksharas")
        self.save_selections_button.clicked.connect(self.save_selections_to_images)
        self.save_selections_button.setEnabled(False)
        selection_layout.addWidget(self.save_selections_button)
        
        self.clear_selection_button = QPushButton("Clear Selections")
        self.clear_selection_button.clicked.connect(self.clear_selection)
        self.clear_selection_button.setEnabled(False)
        selection_layout.addWidget(self.clear_selection_button)

        controls_horizontal_ii.addWidget(selection_group)

        com_group = QGroupBox("Contextual Commentary Corrections")
        com_layout = QHBoxLayout()
        com_layout.setContentsMargins(10, 5, 10, 5)
        com_group.setLayout(com_layout)
        
        self.load_text_button = QPushButton("Load Commentary File")
        self.load_text_button.clicked.connect(self.load_text_to_editor)
        com_layout.addWidget(self.load_text_button)

        self.save_text_button = QPushButton("Save OCR Text")
        self.save_text_button.clicked.connect(self.save_text_from_editor)
        com_layout.addWidget(self.save_text_button)
            
        controls_horizontal_ii.addWidget(com_group)

        main_layout.addLayout(controls_horizontal_ii)

        # Splitter for PDF view and text
        splitter = QSplitter(Qt.Horizontal)

        # PDF view on left side
        self.pdf_scroll_area = QScrollArea()
        self.pdf_scroll_area.setWidgetResizable(True)
        self.pdf_content = PDFViewLabel()
        self.pdf_content.setText("Load a PDF to begin")
        self.pdf_content.setAlignment(Qt.AlignCenter)
        self.pdf_content.selectionChanged.connect(self.update_selection)
        self.pdf_content.selectionFinished.connect(self.finalize_selection)
        self.pdf_scroll_area.setWidget(self.pdf_content)
        splitter.addWidget(self.pdf_scroll_area)

        # Text editor on right side
        self.text_edit = QTextEdit()
        self.text_edit.setReadOnly(False)
        self.text_edit.setPlaceholderText("OCR Text")
        splitter.addWidget(self.text_edit)

        # Set font with Hindi support
        hindi_font = QFont()
        hindi_font.setPointSize(30)
        hindi_font.setFamily("Nunito")
        self.text_edit.setFont(hindi_font)

        # Enable Input Methods for international text input
        self.text_edit.setAttribute(Qt.WA_InputMethodEnabled, True)

        # Set initial sizes for the splitter
        splitter.setSizes([500, 500])

        # Add splitter to main layout
        main_layout.addWidget(splitter, 1)

        # Qwen VL Panel - BELOW THE SPLITTER (IMPROVED LAYOUT)
        qwen_widget = QWidget()
        qwen_layout = QVBoxLayout()
        qwen_widget.setLayout(qwen_layout)
        qwen_widget.setMinimumHeight(240)  # Increased height
        qwen_widget.setMaximumHeight(280)  # Increased maximum height

        # Qwen VL Combined Group Box
        qwen_group = QGroupBox("Qwen VL Vision-Language Model based Visual Question Answering")
        qwen_group_layout = QVBoxLayout()
        qwen_group_layout.setSpacing(10)  # Increased spacing between rows
        qwen_group_layout.setContentsMargins(10, 10, 10, 10)  # Add padding
        qwen_group.setLayout(qwen_group_layout)

        # Row 1 Column 1: Template selection
        row1_layout = QHBoxLayout()
        row1_layout.setSpacing(10)
        row1_layout.addWidget(QLabel("Template:"))
        self.qwen_template_combo = QComboBox()
        self.qwen_template_combo.addItems([
            "Custom Prompt",
            "1. Clutter - Correct OCR",
            "2. Eroded - Correct OCR",
            "3. Single OCR - Correct Aksharas",
            "4. Multi-OCR Consensus",
            "5. Commentary-Based Correction"
        ])
        self.qwen_template_combo.setMinimumWidth(250)
        self.qwen_template_combo.currentIndexChanged.connect(self.update_prompt_from_template)
        row1_layout.addWidget(self.qwen_template_combo)
        
        self.qwen_fill_button = QPushButton("Auto-Fill")
        self.qwen_fill_button.setMaximumWidth(100)
        self.qwen_fill_button.clicked.connect(self.auto_fill_prompt)
        self.qwen_fill_button.setEnabled(False)
        row1_layout.addWidget(self.qwen_fill_button)

        # Row 1 Column 2: Action buttons and status
        self.qwen_selection_button = QPushButton("Analyse Selection")
        self.qwen_selection_button.clicked.connect(self.qwen_analyze_selection)
        self.qwen_selection_button.setEnabled(False)
        self.qwen_selection_button.setMinimumWidth(130)
        row1_layout.addWidget(self.qwen_selection_button)

        self.qwen_stop_button = QPushButton("Stop")
        self.qwen_stop_button.clicked.connect(self.stop_qwen)
        self.qwen_stop_button.setEnabled(False)
        self.qwen_stop_button.setMaximumWidth(70)
        row1_layout.addWidget(self.qwen_stop_button)

        self.qwen_replace_button = QPushButton("Replace Selected Text")
        self.qwen_replace_button.clicked.connect(self.replace_selected_text_with_qwen)
        self.qwen_replace_button.setEnabled(False)
        self.qwen_replace_button.setMinimumWidth(150)
        row1_layout.addWidget(self.qwen_replace_button)
        
        row1_layout.addStretch()
        qwen_group_layout.addLayout(row1_layout)

        # Row 2: Prompt input (single line for cleaner layout)
        row2_layout = QHBoxLayout()
        row2_layout.setSpacing(10)
        
        prompt_label = QLabel("Prompt:")
        prompt_label.setMinimumWidth(70)
        row2_layout.addWidget(prompt_label)
        
        self.qwen_prompt_input = QLineEdit()
        self.qwen_prompt_input.setPlaceholderText("Enter your question or select a template above...")
        self.qwen_prompt_input.setMinimumHeight(30)
        prompt_font = QFont()
        prompt_font.setPointSize(9)
        self.qwen_prompt_input.setFont(prompt_font)
        row2_layout.addWidget(self.qwen_prompt_input)
        
        qwen_group_layout.addLayout(row2_layout)
        
        # Status and progress
        row3_layout = QHBoxLayout()
        row3_layout.setSpacing(10)

        self.qwen_status_label = QLabel("Ready")
        self.qwen_status_label.setMinimumWidth(150)
        self.qwen_status_label.setStyleSheet("color: #666; font-weight: bold;")
        row3_layout.addWidget(self.qwen_status_label)
        
        self.qwen_progress = QProgressBar()
        self.qwen_progress.setRange(0, 100)
        self.qwen_progress.setValue(0)
        self.qwen_progress.setMaximumWidth(200)
        self.qwen_progress.setMaximumHeight(20)
        row3_layout.addWidget(self.qwen_progress)

        row3_layout.addStretch()
        qwen_group_layout.addLayout(row3_layout)

        # Row 4: Results display
        row4_layout = QVBoxLayout()
        results_header_layout = QHBoxLayout()
        results_header_layout.addWidget(QLabel("Results:"))
        
        self.qwen_clear_results_button = QPushButton("Clear Results")
        self.qwen_clear_results_button.clicked.connect(self.clear_qwen_results)
        self.qwen_clear_results_button.setMaximumWidth(100)
        results_header_layout.addWidget(self.qwen_clear_results_button)
        results_header_layout.addStretch()
        
        row4_layout.addLayout(results_header_layout)

        # Results text area (compact, scrollable)
        self.qwen_results_text = QTextEdit()
        self.qwen_results_text.setReadOnly(False)
        self.qwen_results_text.setPlaceholderText("Qwen VL results will appear here...")
        results_font = QFont()
        results_font.setPointSize(10)
        self.qwen_results_text.setFont(results_font)
        self.qwen_results_text.setMinimumHeight(80)  # Increased height
        self.qwen_results_text.setMaximumHeight(100)  # Increased max height
        row4_layout.addWidget(self.qwen_results_text)

        qwen_group_layout.addLayout(row4_layout)

        qwen_layout.addWidget(qwen_group)

        # Add Qwen VL widget below the splitter
        main_layout.addWidget(qwen_widget)

        # Add speech recognition UI
        self.add_speech_recognition_ui()

        # Status bar
        self.statusBar().showMessage("Ready")

    # ==================== AKSHARA SELECTION METHODS ====================
    
    def load_image_lookup_data(self):
        """Load existing image lookup data from JSON file"""
        lookup_file = os.path.join(self.image_lookup_dir, "image_lookup.json")
        
        if os.path.exists(lookup_file):
            try:
                with open(lookup_file, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                
                # Handle both old format (with "images" key) and new format (direct keys)
                if isinstance(data, dict):
                    # Check if it has an "images" key (old format)
                    if "images" in data:
                        self.image_lookup_data = data["images"]
                    else:
                        self.image_lookup_data = data
                else:
                    self.image_lookup_data = {}
                
                # Find the next available ID (only from numeric keys)
                if self.image_lookup_data:
                    numeric_ids = []
                    for k in self.image_lookup_data.keys():
                        try:
                            numeric_ids.append(int(k))
                        except ValueError:
                            # Skip non-numeric keys
                            continue
                    
                    if numeric_ids:
                        max_id = max(numeric_ids)
                        self.next_image_id = max_id + 1
                    else:
                        self.next_image_id = 0
                else:
                    self.next_image_id = 0
                    
                print(f"Loaded {len(self.image_lookup_data)} images from lookup database")
            except Exception as e:
                print(f"Error loading image lookup data: {e}")
                self.image_lookup_data = {}
                self.next_image_id = 0
        else:
            self.image_lookup_data = {}
            self.next_image_id = 0
    
    def save_image_lookup_data(self):
        """Save image lookup data to JSON file"""
        if not os.path.exists(self.image_lookup_dir):
            os.makedirs(self.image_lookup_dir)
        
        lookup_file = os.path.join(self.image_lookup_dir, "image_lookup.json")
        
        try:
            with open(lookup_file, 'w', encoding='utf-8') as f:
                json.dump(self.image_lookup_data, f, indent=2, ensure_ascii=False)
            print(f"Saved {len(self.image_lookup_data)} images to lookup database")
        except Exception as e:
            print(f"Error saving image lookup data: {e}")
    
    def compare_images(self, img1, img2):
        """Compare two images using mean squared error"""
        try:
            FIXED_WIDTH = 100
            FIXED_HEIGHT = 100
            
            img1_resized = img1.scaled(FIXED_WIDTH, FIXED_HEIGHT, Qt.IgnoreAspectRatio, Qt.SmoothTransformation).toImage()
            img2_resized = img2.scaled(FIXED_WIDTH, FIXED_HEIGHT, Qt.IgnoreAspectRatio, Qt.SmoothTransformation).toImage()
            
            img1_array = self.qimage_to_numpy(img1_resized)
            img2_array = self.qimage_to_numpy(img2_resized)
            
            if img1_array.shape != img2_array.shape:
                img2_array = cv2.resize(img2_array, (img1_array.shape[1], img1_array.shape[0]))
            
            err = np.sum((img1_array.astype("float") - img2_array.astype("float")) ** 2)
            err /= float(img1_array.shape[0] * img1_array.shape[1] * img1_array.shape[2])
            
            similarity = 1.0 / (1.0 + err) if err > 0 else 1.0
            return similarity
            
        except Exception as e:
            print(f"Error in image comparison: {str(e)}")
            return 0.01

    def qimage_to_numpy(self, qimage):
        """Convert QImage to numpy array"""
        try:
            width = qimage.width()
            height = qimage.height()
            
            ptr = qimage.constBits()
            ptr.setsize(height * width * 4)
            
            arr = np.frombuffer(ptr, np.uint8).reshape((height, width, 4))
            return arr[:, :, :3]
        except Exception as e:
            print(f"Error in QImage to numpy conversion: {str(e)}")
            return np.zeros((100, 100, 3), dtype=np.uint8)

    def get_similar_images(self, selected_image, num_similar=10):
        """Find similar images in the image lookup database"""
        if not self.image_lookup_data:
            return []
        
        similarities = []
        selected_pixmap = QPixmap.fromImage(selected_image)
        
        for image_id, image_data in self.image_lookup_data.items():
            image_path = image_data.get('path')
            
            if not image_path or not os.path.exists(image_path):
                continue
            
            comparison_pixmap = QPixmap(image_path)
            similarity = self.compare_images(selected_pixmap, comparison_pixmap)
            similarities.append((similarity, comparison_pixmap, image_id))
        
        similarities.sort(reverse=True, key=lambda x: x[0])
        return similarities[:num_similar]

    def save_selections_to_images(self):
        """Save selected regions and show comparison dialog"""
        if not hasattr(self.pdf_content, "user_selections") or not self.pdf_content.user_selections:
            QMessageBox.warning(self, "No Selections", "No character selections to save.")
            return
        
        if not self.curr_page_pixmap:
            return
        
        try:
            if not os.path.exists(self.image_lookup_dir):
                os.makedirs(self.image_lookup_dir)
            
            image = self.curr_page_pixmap.toImage()
            base_filename = os.path.splitext(self.current_pdf_filename)[0]
            if not base_filename:
                base_filename = f"image_page_{self.current_page + 1}"
            
            selected_pixmaps = []
            selected_boxes = []
            
            # Extract all selected images
            for box in self.pdf_content.user_selections:
                x = int(box.x() / self.zoom_factor)
                y = int(box.y() / self.zoom_factor)
                width = int(box.width() / self.zoom_factor)
                height = int(box.height() / self.zoom_factor)
                
                cropped = image.copy(x, y, width, height)
                selected_pixmaps.append(QPixmap.fromImage(cropped))
                selected_boxes.append((box, x, y, width, height))
            
            # Find similar images
            if selected_pixmaps:
                first_image = selected_pixmaps[0].toImage()
                similar_images = self.get_similar_images(first_image, num_similar=10)
                
                if similar_images:
                    # Show comparison dialog
                    dialog = ImageComparisonDialog(self, selected_pixmaps, similar_images)
                    
                    if dialog.exec_() == QDialog.Accepted:
                        results = dialog.get_results()
                        selected_image_ids = results['selected_images']
                        is_insert_underscore = results['insert_underscore']
                        character = results['character']
                        
                        # Insert into text editor
                        self.insert_image_references(selected_image_ids, is_insert_underscore, character)
                        
                        # Save all selected images
                        for i, (box, x, y, width, height) in enumerate(selected_boxes):
                            cropped = image.copy(x, y, width, height)
                            self.save_selected_image(QPixmap.fromImage(cropped), box, f"{base_filename}_{i}")
                else:
                    # No similar images, just save
                    for i, (box, x, y, width, height) in enumerate(selected_boxes):
                        cropped = image.copy(x, y, width, height)
                        self.save_selected_image(QPixmap.fromImage(cropped), box, f"{base_filename}_{i}")
                    
                    QMessageBox.information(
                        self, 
                        "No Similar Images", 
                        f"No similar images found. All {len(selected_boxes)} selections have been saved."
                    )
                
                # Clear selections
                self.pdf_content.clear_user_selections()
                self.pdf_content.update()
                self.save_selections_button.setEnabled(False)
            
        except Exception as e:
            QMessageBox.critical(
                self, 
                "Error Processing Selections", 
                f"An error occurred while processing the selections:\n{str(e)}"
            )

    def save_selected_image(self, cropped_image, box, base_filename):
        """Save the selected image to the image lookup directory"""
        image_filename = f"{base_filename}_char_{self.next_image_id}.png"
        image_path = os.path.join(self.image_lookup_dir, image_filename)
        
        if isinstance(cropped_image, QPixmap):
            cropped_image = cropped_image.toImage()
        
        if cropped_image.save(image_path):
            x = int(box.x() / self.zoom_factor)
            y = int(box.y() / self.zoom_factor)
            width = int(box.width() / self.zoom_factor)
            height = int(box.height() / self.zoom_factor)
            
            self.image_lookup_data[str(self.next_image_id)] = {
                'path': image_path,
                'id': self.next_image_id,
                'char': '$',
                'page': self.current_page + 1,
                'bbox': [x, y, width, height],
                'source_file': self.current_pdf_filename
            }
            
            self.next_image_id += 1
            self.save_image_lookup_data()
            self.statusBar().showMessage(f"Saved image to {image_filename}")
        else:
            self.statusBar().showMessage(f"Failed to save image {image_filename}")

    def insert_image_references(self, image_ids, insert_underscore, character="_"):
        """Insert selected image references at cursor position"""
        if not image_ids:
            return
        
        cursor = self.text_edit.textCursor()
        
        for image_id in image_ids:
            image_data = self.image_lookup_data.get(image_id)
            
            if not image_data:
                continue
            
            if insert_underscore:
                char_to_insert = "_"
            else:
                char_to_insert = character
            
            cursor.insertText(char_to_insert)
        
        self.text_edit.setTextCursor(cursor)
        self.statusBar().showMessage(f"Inserted {len(image_ids)} aksharas")

    # ==================== QWEN VL METHODS ====================
    
    def get_prompt_templates(self):
        """Get prompt templates with placeholders"""
        return {
            0: "",  # Custom
            1: "Given the cluttered image and OCR-predicted text (<ocr_text>), please provide the correct predicted text.",
            2: "Given the eroded image and OCR-predicted text (<ocr_text>), please provide the correct predicted text.",
            3: "Given the OCR-predicted text containing difficult-to-read aksharas: (<ocr_text>), please provide the correct text by correcting the predicted text.",
            4: "Given the image, OCR predicted text 1: (<ocr_text>), and OCR predicted text 2: (<ocr_text_2>), please provide the correct predicted text.",
            5: "Given the image, OCR predicted text: (<ocr_text>), and corresponding commentary of the image: (<commentary>), please provide the correct predicted text."
        }
    
    def update_prompt_from_template(self, index):
        """Update prompt field when template is selected"""
        if index == 0:
            # Custom - clear or keep existing
            return
        
        templates = self.get_prompt_templates()
        template = templates.get(index, "")
        
        if template:
            self.qwen_prompt_input.setText(template)
            # Enable auto-fill button for templates with placeholders
            self.qwen_fill_button.setEnabled(True)
    
    def auto_fill_prompt(self):
        """Auto-fill prompt with current OCR text and other context"""
        prompt = self.qwen_prompt_input.text().strip()
        
        if not prompt:
            QMessageBox.warning(self, "No Template", "Please select a prompt template first.")
            return
        
        # Try to get OCR text from current selection first
        ocr_text = ""
        
        # If there's a selection on the PDF, try to OCR it
        if hasattr(self.pdf_content, 'user_selections') and self.pdf_content.user_selections:
            last_selection = self.pdf_content.user_selections[-1]
            
            # Get OCR cache key for this selection
            bbox_key = (last_selection.x(), last_selection.y(), last_selection.width(), last_selection.height())
            cache_key = (self.current_page, bbox_key)
            
            # Check if we have cached OCR for this selection
            if cache_key in self.ocr_results:
                ocr_text = self.ocr_results[cache_key]
            else:
                # Get OCR from full page text (fallback)
                cursor = self.text_edit.textCursor()
                if cursor.hasSelection():
                    ocr_text = cursor.selectedText().strip()
                else:
                    ocr_text = self.text_edit.toPlainText().strip()
        else:
            # No PDF selection, use text editor content
            cursor = self.text_edit.textCursor()
            if cursor.hasSelection():
                ocr_text = cursor.selectedText().strip()
            else:
                ocr_text = self.text_edit.toPlainText().strip()
        
        if not ocr_text:
            ocr_text = "[No OCR text available]"
        
        # Replace <ocr_text> placeholder
        if "<ocr_text>" in prompt:
            prompt = prompt.replace("<ocr_text>", ocr_text)
        
        # Handle template-specific auto-fills
        template_index = self.qwen_template_combo.currentIndex()
        
        if template_index == 4:  # Multi-OCR Consensus
            # For multi-OCR, try to get alternate OCR if available
            if "<ocr_text_2>" in prompt:
                # Check if we have a second OCR engine result (future enhancement)
                ocr_text_2 = "[Run second OCR or paste alternate prediction here]"
                prompt = prompt.replace("<ocr_text_2>", ocr_text_2)
        
        elif template_index == 5:  # Commentary-based
            # Try to get commentary from somewhere
            if "<commentary>" in prompt:
                # Future enhancement: could load from commentary file
                commentary = "[Paste relevant commentary text here]"
                prompt = prompt.replace("<commentary>", commentary)
        
        # Update the prompt field
        self.qwen_prompt_input.setText(prompt)
        self.statusBar().showMessage("Prompt auto-filled with OCR text and context")
    
    def clear_qwen_results(self):
        """Clear Qwen VL results display"""
        self.qwen_results_text.clear()
        self.qwen_replace_button.setEnabled(False)
        self.statusBar().showMessage("Cleared Qwen VL results")
    
    def qwen_analyze_selection(self):
        """Perform Qwen VL analysis on the selected region"""
        if not self.curr_page_pixmap:
            QMessageBox.warning(self, "No Page", "Please load a PDF page first.")
            return
        
        if not hasattr(self.pdf_content, 'user_selections') or not self.pdf_content.user_selections:
            QMessageBox.warning(self, "No Selection", "Please select a region first using Selection Mode.")
            return
        
        prompt = self.qwen_prompt_input.text().strip()
        if not prompt:
            QMessageBox.warning(self, "No Prompt", "Please enter a question/prompt.")
            return
        
        # Get the last selection
        last_selection = self.pdf_content.user_selections[-1]
        
        # Crop the image to selection
        image = self.curr_page_pixmap.toImage()
        x = int(last_selection.x() / self.zoom_factor)
        y = int(last_selection.y() / self.zoom_factor)
        width = int(last_selection.width() / self.zoom_factor)
        height = int(last_selection.height() / self.zoom_factor)
        
        cropped = image.copy(x, y, width, height)
        cropped_pixmap = QPixmap.fromImage(cropped)
        
        self.start_qwen_analysis(cropped_pixmap, prompt)
    
    def start_qwen_analysis(self, pixmap, prompt):
        """Start Qwen VL analysis in background thread"""
        # Update UI
        self.qwen_selection_button.setEnabled(False)
        self.qwen_stop_button.setEnabled(True)
        self.qwen_replace_button.setEnabled(False)
        self.qwen_progress.setRange(0, 100)
        self.qwen_progress.setValue(0)
        self.qwen_status_label.setText("Starting analysis...")
        
        # Create worker
        worker = QwenVLWorker(pixmap, prompt, self.qwen_server_url)
        worker.result_ready.connect(self.handle_qwen_result)
        worker.error_occurred.connect(self.handle_qwen_error)
        worker.progress_update.connect(self.update_qwen_progress)
        
        self.qwen_workers.append(worker)
        worker.start()
        
        self.statusBar().showMessage(f"Qwen VL analysis started: {prompt[:50]}...")
    
    def handle_qwen_result(self, answer, prompt):
        """Handle Qwen VL result"""
        # Update UI
        self.qwen_selection_button.setEnabled(True)
        self.qwen_stop_button.setEnabled(False)
        self.qwen_progress.setValue(100)
        self.qwen_status_label.setText("Analysis complete!")
        self.qwen_replace_button.setEnabled(True)
        
        # Display result
        result_text = f"**Question:** {prompt}\n\n**Answer:**\n{answer}\n\n{'='*50}\n\n"
        
        # Append to existing results
        current_text = self.qwen_results_text.toPlainText()
        self.qwen_results_text.setText(result_text + current_text)
        
        self.statusBar().showMessage("Qwen VL analysis completed")
        
        # Clean up worker
        for worker in self.qwen_workers[:]:
            if worker.isFinished():
                self.qwen_workers.remove(worker)
    
    def handle_qwen_error(self, error_message):
        """Handle Qwen VL error"""
        # Update UI
        self.qwen_selection_button.setEnabled(True)
        self.qwen_stop_button.setEnabled(False)
        self.qwen_progress.setValue(0)
        self.qwen_status_label.setText(f"Error")
        
        QMessageBox.critical(self, "Qwen VL Error", f"An error occurred:\n\n{error_message}")
        self.statusBar().showMessage(f"Qwen VL error: {error_message}")
    
    def update_qwen_progress(self, progress, status_message):
        """Update Qwen VL progress"""
        self.qwen_progress.setValue(progress)
        self.qwen_status_label.setText(status_message)
        self.statusBar().showMessage(status_message)
    
    def stop_qwen(self):
        """Stop current Qwen VL analysis"""
        for worker in self.qwen_workers:
            if worker.isRunning():
                worker.stop()
                worker.quit()
                worker.wait()
        
        self.qwen_selection_button.setEnabled(True)
        self.qwen_stop_button.setEnabled(False)
        self.qwen_progress.setValue(0)
        self.qwen_status_label.setText("Stopped")
        self.statusBar().showMessage("Qwen VL analysis stopped")
    
    def replace_selected_text_with_qwen(self):
        """Replace selected text in editor with Qwen VL results"""
        qwen_text = self.qwen_results_text.toPlainText()
        
        if not qwen_text:
            QMessageBox.information(self, "No Results", "No Qwen VL results to insert.")
            return
        
        # Extract just the answer part (skip the question)
        lines = qwen_text.split('\n')
        answer_lines = []
        in_answer = False
        
        for line in lines:
            if line.startswith("**Answer:**"):
                in_answer = True
                continue
            elif line.startswith("==="):
                break
            elif in_answer and line.strip():
                answer_lines.append(line)
        
        answer_text = '\n'.join(answer_lines).strip()
        
        if not answer_text:
            answer_text = qwen_text  # Fallback to full text
        
        # Get current cursor or selection
        cursor = self.text_edit.textCursor()
        
        if cursor.hasSelection():
            # Replace selected text
            cursor.removeSelectedText()
        
        # Insert Qwen VL result
        cursor.insertText(answer_text)
        self.text_edit.setTextCursor(cursor)
        
        self.statusBar().showMessage("Replaced text with Qwen VL result")

    def toggle_selection_mode(self, state):
        """Toggle selection mode on/off"""
        enabled = (state == Qt.Checked)
        self.pdf_content.enable_selection(enabled)
        
        if enabled:
            self.statusBar().showMessage("Selection mode enabled - Click and drag to select aksharas")
        else:
            self.statusBar().showMessage("Selection mode disabled")

    def update_selection(self, rect):
        """Update selection display"""
        pass

    def finalize_selection(self, rect):
        """Finalize a selection"""
        if self.pdf_content.selection_mode and not rect.isEmpty():
            if self.pdf_content.add_user_selection(rect):
                self.save_selections_button.setEnabled(True)
                self.clear_selection_button.setEnabled(True)
                self.qwen_selection_button.setEnabled(True)
                
                # Enable auto-fill if a template is selected
                if self.qwen_template_combo.currentIndex() > 0:
                    self.qwen_fill_button.setEnabled(True)
                
                count = len(self.pdf_content.user_selections)
                self.statusBar().showMessage(f"Added selection ({count} total)")

    def clear_selection(self):
        """Clear all selections"""
        if hasattr(self.pdf_content, 'clear_user_selections'):
            self.pdf_content.clear_user_selections()
            self.save_selections_button.setEnabled(False)
            self.clear_selection_button.setEnabled(False)
            self.qwen_selection_button.setEnabled(False)
            self.statusBar().showMessage("Cleared all selections")

    # ==================== OCR METHODS ====================
    
    def perform_ocr_on_current_page(self):
        """Perform OCR on the current page automatically"""
        if not self.curr_page_pixmap:
            return
        
        page_num = self.current_page
        
        worker = OCRWorker(self.curr_page_pixmap, page_num, None, self.ocr_language)
        worker.result_ready.connect(self.handle_ocr_result)
        worker.error_occurred.connect(self.handle_ocr_error)
        
        self.ocr_workers.append(worker)
        worker.start()
        
        self.statusBar().showMessage(f"Performing OCR on page {page_num + 1}...")

    def handle_ocr_result(self, text, page_num, bbox=None):
        """Handle OCR result"""
        bbox_key = None
        if bbox and not bbox.isEmpty():
            bbox_key = (bbox.x(), bbox.y(), bbox.width(), bbox.height())
        
        cache_key = (page_num, bbox_key)
        self.ocr_results[cache_key] = text
        
        if page_num == self.current_page:
            self.text_edit.setText(text)
            self.statusBar().showMessage(f"OCR completed for page {page_num + 1} ({len(text)} characters)")
        
        for worker in self.ocr_workers[:]:
            if worker.isFinished():
                self.ocr_workers.remove(worker)

    def handle_ocr_error(self, error_message):
        """Handle OCR error"""
        self.statusBar().showMessage(f"OCR Error: {error_message}")

    # ==================== HINDI TYPING SUPPORT ====================
    
    def change_keyboard_layout(self, index):
        """Switch between keyboard layouts"""
        if index == 0:
            self.statusBar().showMessage("Switched to Hindi transliteration mode")
            self.text_edit.setFocus()
            self.hindi_help_button.setEnabled(True)
        else:
            self.statusBar().showMessage("Switched to English keyboard")
            self.hindi_help_button.setEnabled(False)

    def show_hindi_typing_help(self):
        """Show a help dialog for Hindi typing"""
        help_text = """
    <h3>हिंदी में टाइपिंग के लिए मदद</h3>
    <p>आप अंग्रेजी अक्षरों का उपयोग करके हिंदी में लिख सकते हैं:</p>
    <ul>
        <li><b>namaste</b> लिखने पर <b>नमस्ते</b> बनेगा</li>
        <li><b>main</b> लिखने पर <b>मैं</b> बनेगा</li>
        <li><b>hindi</b> लिखने पर <b>हिंदी</b> बनेगा</li>
    </ul>
    <p>हिंदी और अंग्रेजी अक्षर: मैपिंग</p>
    <ul>
        <li>a = अ | ā = आ</li>
        <li>i = इ | ī = ई</li>
        <li>u = उ | ū = ऊ</li>
        <li>o = ओ | e = ए</li>
        <li>ṛ = ऋ | ai = ऐ</li>
        <li>au = औ | ka = क</li>
        <li>kha = ख | ga = ग</li>
        <li>gha = घ | ṅa = ङ</li>
        <li>ca = च | cha = छ</li>
        <li>ja = ज | jha = झ</li>
        <li>ña = ञ | ṭa = ट</li>
        <li>ṭha = ठ | ḍa = ड</li>
        <li>ḍha = ढ | ṇa = ण</li>
        <li>ta = त | tha = थ</li>
        <li>da = द | dha = ध</li>
        <li>na = न | pha = फ</li>
        <li>ba = ब | bha = भ</li>
        <li>ma = म | ya = य</li>
        <li>ra = र | la = ल</li>
        <li>va = व | śa = श</li>
        <li>ṣa = ष | sa = स</li>
        <li>ha = ह | pa = प</li>
    </ul>
    <p>आप अपने सिस्टम इनपुट सेटिंग्स से भी हिंदी कीबोर्ड सक्षम कर सकते हैं।</p>
        """
        
        msg_box = QMessageBox(self)
        msg_box.setWindowTitle("Hindi Typing Help")
        msg_box.setTextFormat(Qt.RichText)
        msg_box.setText(help_text)
        msg_box.setStandardButtons(QMessageBox.Ok)
        msg_box.exec_()

    def set_hindi_mode(self):
        """Set the text editor to Hindi input mode"""
        if hasattr(self, 'keyboard_combo'):
            self.keyboard_combo.setCurrentIndex(0)
        
        self.text_edit.setFocus()
        self.statusBar().showMessage("Hindi input mode enabled")

    def init_text_editor_context_menu(self):
        """Initialize custom context menu for text editor"""
        self.text_edit.setContextMenuPolicy(Qt.CustomContextMenu)
        self.text_edit.customContextMenuRequested.connect(self.show_text_editor_context_menu)

    def show_text_editor_context_menu(self, position):
        """Show custom context menu for text editor"""
        menu = self.text_edit.createStandardContextMenu(position)
        menu.addSeparator()
        
        hindi_action = menu.addAction("Enable Hindi Typing")
        hindi_action.triggered.connect(self.set_hindi_mode)
        
        hindi_help_action = menu.addAction("Hindi Typing Help")
        hindi_help_action.triggered.connect(self.show_hindi_typing_help)
        
        menu.exec_(self.text_edit.mapToGlobal(position))

    def load_text_to_editor(self):
        """Load text from a file"""
        file_path, _ = QFileDialog.getOpenFileName(
            self, "Load Text File", "", "Text Files (*.txt);;All Files (*.*)")
        
        if not file_path:
            return
            
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                text = f.read()
            
            cursor = self.text_edit.textCursor()
            cursor.insertText(text)
            
            self.statusBar().showMessage(f"Text loaded from {os.path.basename(file_path)}")
        except Exception as e:
            QMessageBox.critical(self, "Error Loading File", f"An error occurred:\n{str(e)}")

    def save_text_from_editor(self):
        """Save the text from the editor"""
        text = self.text_edit.toPlainText()
        if not text:
            QMessageBox.warning(self, "No Text", "There is no text to save.")
            return
        
        default_filename = "Text_"
        if self.current_pdf_filename:
            default_filename += os.path.splitext(self.current_pdf_filename)[0] + "_"
        default_filename += f"page_{self.current_page + 1}.txt"
        
        file_path, _ = QFileDialog.getSaveFileName(
            self, "Save Text", default_filename, "Text Files (*.txt);;All Files (*.*)")
        
        if not file_path:
            return
            
        try:
            with open(file_path, 'w', encoding='utf-8') as f:
                f.write(text)
            
            self.statusBar().showMessage(f"Text saved to {os.path.basename(file_path)}")
        except Exception as e:
            QMessageBox.critical(self, "Error Saving File", f"An error occurred:\n{str(e)}")

    # ==================== PDF NAVIGATION ====================
    
    def load_pdf(self):
        file_path, _ = QFileDialog.getOpenFileName(
            self, "Open PDF File", "", "PDF Files (*.pdf)")
        
        if not file_path:
            return
            
        try:
            self.pdf_document = fitz.open(file_path)
            self.total_pages = len(self.pdf_document)
            self.current_page = 0
            self.current_pdf_filename = os.path.basename(file_path)
            
            self.page_label.setText(f"Page {self.current_page + 1} of {self.total_pages}")
            self.prev_button.setEnabled(False)
            self.next_button.setEnabled(self.total_pages > 1)
            
            # Enable controls
            self.zoom_out_button.setEnabled(True)
            self.zoom_in_button.setEnabled(True)
            self.zoom_slider.setEnabled(True)
            self.zoom_reset_button.setEnabled(True)
            self.selection_checkbox.setEnabled(True)
            self.save_text_button.setEnabled(True)
            
            # Reset zoom
            self.zoom_reset()
            
            # Clear selection
            self.clear_selection()
            
            # Load first page
            self.load_page(0)
            
            self.statusBar().showMessage(f"Loaded: {self.current_pdf_filename} ({self.total_pages} pages)")
        
        except Exception as e:
            self.statusBar().showMessage(f"Error loading PDF: {str(e)}")
    
    def load_page(self, page_num):
        if not self.pdf_document or page_num < 0 or page_num >= self.total_pages:
            return
        
        self.clear_selection()
        
        page = self.pdf_document.load_page(page_num)
        zoom_matrix = fitz.Matrix(2.0, 2.0)
        pix = page.get_pixmap(matrix=zoom_matrix)
        
        # Convert to numpy array for binarization
        img_data = pix.tobytes("ppm")
        qimg = QImage.fromData(img_data)
        
        # Convert QImage to numpy
        width = qimg.width()
        height = qimg.height()
        ptr = qimg.constBits()
        ptr.setsize(height * width * 4)
        arr = np.frombuffer(ptr, np.uint8).reshape((height, width, 4))
        
        # Convert to grayscale
        gray = cv2.cvtColor(arr[:, :, :3], cv2.COLOR_RGB2GRAY)
        
        # Apply binary threshold (Otsu's method)
        _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        
        # Convert back to QImage
        binary_qimg = QImage(binary.data, width, height, width, QImage.Format_Grayscale8)
        binary_qimg.bits().setsize(height * width)
        
        # Convert to QPixmap
        pixmap = QPixmap.fromImage(binary_qimg)
        
        self.curr_page_pixmap = pixmap
        self.apply_zoom()
        
        self.current_page = page_num
        self.page_label.setText(f"Page {page_num + 1} of {self.total_pages}")
        
        self.prev_button.setEnabled(page_num > 0)
        self.next_button.setEnabled(page_num < self.total_pages - 1)
        
        if hasattr(self.pdf_content, "clear_bounding_boxes"):
            self.pdf_content.clear_bounding_boxes()
        
        # Check for cached OCR results
        cache_key = (page_num, None)
        if cache_key in self.ocr_results:
            self.text_edit.setText(self.ocr_results[cache_key])
            self.statusBar().showMessage(f"Loaded cached OCR for page {page_num + 1}")
        else:
            self.perform_ocr_on_current_page()
    
    def next_page(self):
        if self.current_page < self.total_pages - 1:
            self.load_page(self.current_page + 1)
    
    def prev_page(self):
        if self.current_page > 0:
            self.load_page(self.current_page - 1)
    
    def wheelEvent(self, event):
        """Handle mouse wheel for zooming"""
        if event.modifiers() & Qt.ControlModifier:
            delta = event.angleDelta().y()
            if delta > 0:
                self.zoom_in()
            else:
                self.zoom_out()
            event.accept()
        else:
            super().wheelEvent(event)
    
    # ==================== ZOOM CONTROLS ====================
    
    def zoom_in(self):
        if not self.zoom_slider.isEnabled():
            return
        new_value = min(self.zoom_slider.value() + 25, self.zoom_slider.maximum())
        self.zoom_slider.setValue(new_value)
    
    def zoom_out(self):
        if not self.zoom_slider.isEnabled():
            return
        new_value = max(self.zoom_slider.value() - 25, self.zoom_slider.minimum())
        self.zoom_slider.setValue(new_value)
    
    def zoom_reset(self):
        if not self.zoom_slider.isEnabled():
            return
        self.zoom_slider.setValue(35)
    
    def zoom_slider_changed(self, value):
        self.zoom_factor = value / 100.0
        self.zoom_label.setText(f"{value}%")
        self.apply_zoom()
    
    def apply_zoom(self):
        if not self.curr_page_pixmap:
            return
        
        scaled_pixmap = self.curr_page_pixmap.scaled(
            int(self.curr_page_pixmap.width() * self.zoom_factor),
            int(self.curr_page_pixmap.height() * self.zoom_factor),
            Qt.KeepAspectRatio,
            Qt.SmoothTransformation
        )
        
        self.pdf_content.setPixmap(scaled_pixmap)
        self.pdf_content.setFixedSize(scaled_pixmap.size())

    # ==================== SPEECH RECOGNITION ====================
    
    def add_speech_recognition_ui(self):
        """Add speech recognition UI elements (COMPACT VERSION)"""
        typing_group = QGroupBox("Speech & Keyboard Input")
        typing_layout = QHBoxLayout()
        typing_group.setLayout(typing_layout)
        typing_group.setMaximumHeight(80)
        
        # Speech recognition controls
        self.speech_button = QPushButton("Start Speech")
        self.speech_button.clicked.connect(self.start_speech_recognition)
        typing_layout.addWidget(self.speech_button)
        
        self.speech_stop_button = QPushButton("Stop")
        self.speech_stop_button.clicked.connect(self.stop_speech_recognition)
        self.speech_stop_button.setEnabled(False)
        typing_layout.addWidget(self.speech_stop_button)
        
        self.speech_progress = QProgressBar()
        self.speech_progress.setRange(0, 100)
        self.speech_progress.setValue(0)
        self.speech_progress.setMaximumWidth(150)
        typing_layout.addWidget(self.speech_progress)
        
        typing_layout.addWidget(QLabel("Speech Lang:"))
        self.speech_lang_combo = QComboBox()
        self.speech_lang_combo.addItems(["Hindi", "English (US)"])
        self.speech_lang_combo.setCurrentIndex(0)
        self.speech_lang_combo.currentIndexChanged.connect(self.change_speech_language)
        self.speech_lang_combo.setMaximumWidth(120)
        typing_layout.addWidget(self.speech_lang_combo)
        
        typing_layout.addWidget(QLabel("Keyboard:"))
        self.keyboard_combo = QComboBox()
        self.keyboard_combo.addItems(["Hindi (Transliteration)", "English"])
        self.keyboard_combo.setCurrentIndex(0)
        self.keyboard_combo.currentIndexChanged.connect(self.change_keyboard_layout)
        self.keyboard_combo.setMaximumWidth(180)
        typing_layout.addWidget(self.keyboard_combo)

        self.hindi_help_button = QPushButton("Hindi Typing Help")
        self.hindi_help_button.clicked.connect(self.show_hindi_typing_help)
        self.hindi_help_button.setEnabled(True)
        typing_layout.addWidget(self.hindi_help_button)
        
        typing_layout.addStretch()
        
        self.main_layout.addWidget(typing_group)

    def init_speech_recognition(self):
        """Initialize speech recognition worker"""
        self.speech_worker = None
        self.speech_language = "hi-IN"
        self.speech_timeout = 10
        self.speech_phrase_limit = 10

    def start_speech_recognition(self):
        """Start speech recognition"""
        if self.speech_worker and self.speech_worker.listening:
            self.statusBar().showMessage("Already listening...")
            return
        
        self.speech_button.setEnabled(False)
        self.speech_stop_button.setEnabled(True)
        self.speech_progress.setRange(0, 0)
        
        self.speech_worker = SpeechRecognitionWorker(
            language=self.speech_language,
            timeout=self.speech_timeout,
            phrase_time_limit=self.speech_phrase_limit
        )
        
        self.speech_worker.result_ready.connect(self.handle_speech_result)
        self.speech_worker.error_occurred.connect(self.handle_speech_error)
        self.speech_worker.status_update.connect(self.update_speech_status)
        
        self.speech_worker.start()

    def stop_speech_recognition(self):
        """Stop speech recognition"""
        if self.speech_worker and self.speech_worker.listening:
            self.speech_worker.terminate()
            
        self.speech_button.setEnabled(True)
        self.speech_stop_button.setEnabled(False)
        self.speech_progress.setRange(0, 100)
        self.speech_progress.setValue(0)
        self.statusBar().showMessage("Speech recognition stopped")
    
    def handle_speech_result(self, text):
        """Handle recognized speech"""
        self.speech_button.setEnabled(True)
        self.speech_stop_button.setEnabled(False)
        self.speech_progress.setRange(0, 100)
        self.speech_progress.setValue(100)
        
        if text:
            cursor = self.text_edit.textCursor()
            cursor.insertText(text + " ")
            self.text_edit.setTextCursor(cursor)
            self.text_edit.setFocus()
            self.statusBar().showMessage(f"Recognized: {text}")
        else:
            self.statusBar().showMessage("No speech recognized")

    def handle_speech_error(self, error_message):
        """Handle speech recognition errors"""
        self.speech_button.setEnabled(True)
        self.speech_stop_button.setEnabled(False)
        self.speech_progress.setRange(0, 100)
        self.speech_progress.setValue(0)
        self.statusBar().showMessage(error_message)

    def update_speech_status(self, status_message):
        """Update status during speech recognition"""
        self.statusBar().showMessage(status_message)

    def change_speech_language(self, index):
        """Change speech recognition language"""
        languages = {
            0: "hi-IN",
            1: "en-US"
        }
        self.speech_language = languages.get(index, "hi-IN")
        self.statusBar().showMessage(f"Speech language: {self.speech_language}")

    # ==================== CLEANUP ====================
    
    def closeEvent(self, event):
        """Clean up on close"""
        # Save image lookup data
        if self.image_lookup_data:
            self.save_image_lookup_data()
        
        # Clean up speech recognition worker
        if hasattr(self, 'speech_worker') and self.speech_worker:
            self.speech_worker.quit()
            self.speech_worker.wait()
        
        # Clean up OCR workers
        for worker in self.ocr_workers:
            worker.stop()
            worker.quit()
            worker.wait()
        
        # Clean up Qwen VL workers
        for worker in self.qwen_workers:
            worker.stop()
            worker.quit()
            worker.wait()
        
        if self.pdf_document:
            self.pdf_document.close()
        
        super().closeEvent(event)

# ==================== MAIN ====================

if __name__ == "__main__":
    app = QApplication(sys.argv)
    viewer = PDFOCRViewer()
    viewer.show()
    sys.exit(app.exec_())

Loaded 12 images from lookup database


## Will have to add functionality to add custom tesseract loading